# Prithvi flood-extent inference

This notebook is limited to the model workflow: Sentinel-2 preparation, Prithvi-100M-sen1floods11 inference, validation, GeoJSON conversion, and a provenance-complete `HazardEvent` fixture.

**Scope:** water/flood segmentation only. Do not interpret this model output as landslide detection or operational flood verification.

**Input band order:** Blue (B02), Green (B03), Red (B04), Narrow NIR (B8A), SWIR1 (B11), SWIR2 (B12). The model expects a single six-band GeoTIFF in exactly this order.

In [22]:
# Colab setup: keep its managed Jupyter kernel unchanged.
# The legacy model itself runs in this isolated Python 3.9 environment.
import sys
from pathlib import Path

WORKDIR = Path('/content/prithvi_flood')
WORKDIR.mkdir(parents=True, exist_ok=True)
PY39 = Path('/content/prithvi39/bin/python')
if not PY39.exists():
    !sudo apt-get update -qq
    !sudo apt-get install -y -qq python3.9 python3.9-venv
    !python3.9 -m venv /content/prithvi39

!{PY39} -m pip install -q --upgrade pip setuptools wheel
!{sys.executable} -m pip install -q rasterio shapely matplotlib huggingface_hub
print('Notebook kernel:', sys.version.split()[0])
!{PY39} --version
print('Working directory:', WORKDIR)

3.9.25 (main, Nov  7 2025, 18:07:57) 
[GCC 11.4.0]
IPYKERNEL OK


In [23]:
!nvidia-smi
print('If this shows Tesla T4, the Colab GPU is ready. If not, use Runtime > Change runtime type > T4 GPU, then reconnect.')

Fri Aug 14 14:10:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# Intentionally no kernel replacement. Modifying Colab's managed kernel can crash the session.
print('Using the isolated Python 3.9 environment only for legacy inference.')

usage: jupyter-kernelspec
       [-h]
       [extra_args ...]
jupyter-kernelspec: error: unrecognized arguments: /usr/bin/python3.9
{"argv": ["/usr/bin/python3.9", "-m", "ipykernel_launcher", "-f", "{connection_file}"], "display_name": "Python 3", "language": "python"}


In [2]:
import sys; print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [3]:
print('Legacy inference interpreter:', PY39)
!{PY39} -c "import sys; print(sys.version)"

Available kernels:
  julia        /root/.local/share/jupyter/kernels/julia
  prithvi39    /root/.local/share/jupyter/kernels/prithvi39
  ir           /usr/local/share/jupyter/kernels/ir
  python3      /usr/local/share/jupyter/kernels/python3


In [1]:
# Do not alter /usr/bin/python3 or Jupyter kernelspecs in Colab.
print('Safe runtime configuration confirmed.')

{"argv": ["/usr/bin/python3.9", "-m", "ipykernel_launcher", "-f", "{connection_file}"], "display_name": "Python 3", "language": "python"}


In [ ]:
import sys; print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [ ]:
# The Python 3.9 environment is separate, so Colab's own Python stays supported.
!{PY39} --version


Python 3.9.25


In [ ]:
import sys; print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


: 

: 

## 0. Runtime requirements

The original Prithvi-100M Sen1Floods11 inference stack uses the legacy MMSegmentation/MMCV APIs. In Colab, the notebook keeps the managed notebook kernel intact and creates an isolated **Python 3.9** environment at `/content/prithvi39` for model installation and inference. Do not mix this legacy stack with TerraTorch or the newer Prithvi-EO-2.0 stack.

In [18]:
assert PY39.exists(), 'Run the Colab setup cell first.'
print('Notebook work:', sys.version.split()[0])
print('Model inference:', end=' ')
!{PY39} -c "import sys; assert sys.version_info[:2] == (3, 9); print(sys.version)"
print('Working directory:', WORKDIR)

AssertionError: Use a fresh Python 3.9 GPU environment for this legacy Prithvi-100M inference stack.

## 1. Install the official legacy inference dependencies

This follows the NASA IMPACT `hls-foundation-os` inference path used by the model card. Run once per new Colab session. The final command verifies that the isolated Python 3.9 environment can see the selected GPU.

In [ ]:
%cd /content
!test -d hls-foundation-os || git clone --depth 1 https://github.com/NASA-IMPACT/hls-foundation-os.git
%cd /content/hls-foundation-os
!{PY39} -m pip install -q torch==1.11.0+cu115 torchvision==0.12.0+cu115 -f https://download.pytorch.org/whl/torch_stable.html
!{PY39} -m pip install -q -e .
!{PY39} -m pip install -q -U openmim
# Select the wheel URL that matches the CUDA and torch versions in this environment.
!{PY39} -m mim install 'mmcv-full==1.6.2' -f https://download.openmmlab.com/mmcv/dist/cu115/torch1.11.0/index.html
!{PY39} -m pip install -q 'mmsegmentation==0.30.0' rasterio shapely matplotlib huggingface_hub
!{PY39} -c "import torch; assert torch.cuda.is_available(), 'No GPU: select T4 GPU in Colab and reconnect'; print(torch.__version__, torch.cuda.get_device_name(0))"

## 2. Locate and unpack the Sentinel-2 L2A scene

The next cell mounts Google Drive when running in Colab. Put the uploaded `.SAFE.zip` in `MyDrive/terracascade/` or change `SCENE_ZIP_PATH`. The notebook discovers the six required band files rather than relying on pasted paths.

In [ ]:
# Colab asks you to authorize Drive access once per session.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Colab; use a local path for SCENE_ZIP_PATH.')

# Upload the .SAFE.zip to this Drive folder, or replace this path.
SCENE_ZIP_PATH = Path('/content/drive/MyDrive/terracascade/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE.zip')
assert SCENE_ZIP_PATH.exists(), f'Missing scene archive: {SCENE_ZIP_PATH}'

import shutil
SCENE_DIR = WORKDIR / 'scene'
if not SCENE_DIR.exists():
    shutil.unpack_archive(SCENE_ZIP_PATH, WORKDIR)
    extracted = next(WORKDIR.glob('*.SAFE'))
    extracted.rename(SCENE_DIR)

def find_one(pattern):
    matches = list(SCENE_DIR.rglob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {pattern}; found {len(matches)}: {matches}')
    return matches[0]

band_paths = {
    'B02': find_one('*_B02_10m.jp2'), 'B03': find_one('*_B03_10m.jp2'),
    'B04': find_one('*_B04_10m.jp2'), 'B8A': find_one('*_B8A_20m.jp2'),
    'B11': find_one('*_B11_20m.jp2'), 'B12': find_one('*_B12_20m.jp2'),
}
band_paths

## 3. Build a six-band GeoTIFF for the AOI

Enter the AOI bounds in WGS84. Crop before inference: a full Sentinel-2 tile is unnecessarily large and can exhaust GPU memory. The output is explicitly written with the GeoTIFF driver; copying a JP2 profile was the cause of the previous write failure.

In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds, reproject

# Replace with the reviewed AOI boundary: west, south, east, north (EPSG:4326).
AOI_NAME = 'Idamalayar AOI'
AOI_BOUNDS_WGS84 = (76.65, 10.14, 76.80, 10.36)
BAND_ORDER = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']

with rasterio.open(band_paths['B02']) as ref_src:
    bounds = transform_bounds('EPSG:4326', ref_src.crs, *AOI_BOUNDS_WGS84, densify_pts=21)
    ref_window = from_bounds(*bounds, transform=ref_src.transform).round_offsets().round_lengths()
    ref_transform = ref_src.window_transform(ref_window)
    ref_crs = ref_src.crs
    height, width = int(ref_window.height), int(ref_window.width)
    assert height > 0 and width > 0, 'AOI does not overlap the Sentinel-2 scene'

stack = np.zeros((6, height, width), dtype=np.float32)
for i, band in enumerate(BAND_ORDER):
    with rasterio.open(band_paths[band]) as src:
        reproject(
            source=rasterio.band(src, 1), destination=stack[i],
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_transform, dst_crs=ref_crs,
            dst_shape=(height, width), resampling=Resampling.bilinear,
        )

STACKED_TIF = WORKDIR / 'idamalayar_6band_input.tif'
profile = {'driver': 'GTiff', 'height': height, 'width': width, 'count': 6, 'dtype': 'float32',
           'crs': ref_crs, 'transform': ref_transform, 'compress': 'deflate'}
with rasterio.open(STACKED_TIF, 'w', **profile) as dst:
    dst.write(stack)
    dst.descriptions = tuple(BAND_ORDER)
print(STACKED_TIF, stack.shape, BAND_ORDER)

In [ ]:
import matplotlib.pyplot as plt
rgb = stack[[2, 1, 0]]
rgb = np.clip(rgb / np.percentile(rgb, 98), 0, 1).transpose(1, 2, 0)
plt.figure(figsize=(8, 8)); plt.imshow(rgb); plt.title('RGB band-order check'); plt.axis('off');
assert rasterio.open(STACKED_TIF).count == 6

## 4. Download the exact model files and run the official inference script

The legacy repository name redirects to `ibm-nasa-geospatial/Prithvi-EO-1.0-100M-sen1floods11`. Save the resolved repository and file paths in the provenance record. The model output labels are 0 = no water, 1 = water/flood, -1 = no data/cloud.

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_REPOSITORY = 'ibm-nasa-geospatial/Prithvi-EO-1.0-100M-sen1floods11'
config_path = hf_hub_download(MODEL_REPOSITORY, 'sen1floods11_Prithvi_100M.py')
checkpoint_path = hf_hub_download(MODEL_REPOSITORY, 'sen1floods11_Prithvi_100M.pth')
print(config_path, checkpoint_path, sep='\n')

INPUT_DIR, OUTPUT_DIR = WORKDIR / 'inference_input', WORKDIR / 'inference_output'
INPUT_DIR.mkdir(exist_ok=True); OUTPUT_DIR.mkdir(exist_ok=True)
shutil.copy2(STACKED_TIF, INPUT_DIR / STACKED_TIF.name)
%cd /content/hls-foundation-os
!{PY39} model_inference.py -config {config_path} -ckpt {checkpoint_path} -input {INPUT_DIR} -output {OUTPUT_DIR} -input_type tif -bands 0 1 2 3 4 5
print(list(OUTPUT_DIR.rglob('*')))

## 5. Validate the mask, polygonise flood pixels, and create the fixture

Set `MASK_TIF` to the raster emitted by `model_inference.py`. Inspect the overlay before accepting the result. Remove very small polygons to avoid pixel-scale noise; this threshold is a demo simplification, not model confidence.

In [ ]:
from datetime import datetime, timezone
import json
from shapely.geometry import shape, mapping
from rasterio.features import shapes

MASK_TIF = next(OUTPUT_DIR.rglob('*.tif'))  # verify this is the predicted-label raster, not a visualisation
with rasterio.open(MASK_TIF) as src:
    mask = src.read(1)
    assert mask.shape == (height, width), (mask.shape, (height, width))

plt.figure(figsize=(8, 8)); plt.imshow(rgb); plt.imshow(np.ma.masked_where(mask != 1, mask), cmap='Blues', alpha=.55); plt.title('Prithvi water/flood mask overlay'); plt.axis('off');

MIN_POLYGON_AREA_M2 = 2_500
features = []
for geometry, value in shapes((mask == 1).astype('uint8'), mask=(mask == 1), transform=ref_transform):
    polygon = shape(geometry)
    if polygon.area >= MIN_POLYGON_AREA_M2:
        feature_id = f'flood-zone-{len(features) + 1:03d}'
        features.append({'type': 'Feature', 'id': feature_id, 'properties': {'id': feature_id}, 'geometry': mapping(polygon)})

flood_extent = {'type': 'FeatureCollection', 'features': features}
assert features, 'No polygons remain: verify the predicted mask and AOI before continuing.'
(WORKDIR / 'flood_extent.geojson').write_text(json.dumps(flood_extent, indent=2))
print(f'{len(features)} flood polygons written')

In [ ]:
SCENE_ID = 'S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431'
SCENE_ACQUIRED_AT = '2026-08-10T05:06:49Z'
RUN_AT = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
hazard_event = {
    'id': 'flood-idamalayar-20260810', 'hazard': 'flood', 'severity': 'orange',
    'source': f'Prithvi-100M-sen1floods11 inference ({MODEL_REPOSITORY}), Sentinel-2 L2A scene {SCENE_ID} acquired {SCENE_ACQUIRED_AT}, {AOI_NAME}',
    'status': 'verified-demo', 'issuedAt': SCENE_ACQUIRED_AT,
    'affectedZones': [feature['id'] for feature in features],
    'limitations': ['single-timestamp inference', 'not a live feed', 'demo AOI only', 'cloud-sensitive optical input; not ground-truthed'],
}
provenance = {
    'model': 'Prithvi-100M-sen1floods11', 'resolvedModelRepository': MODEL_REPOSITORY,
    'configPath': str(config_path), 'checkpointPath': str(checkpoint_path),
    'inputBands': BAND_ORDER, 'sceneId': SCENE_ID, 'sceneAcquiredAt': SCENE_ACQUIRED_AT,
    'aoiName': AOI_NAME, 'aoiBoundsWgs84': AOI_BOUNDS_WGS84, 'runAt': RUN_AT,
    'maskFile': str(MASK_TIF), 'minimumPolygonAreaM2': MIN_POLYGON_AREA_M2,
}
fixture = {'hazardEvent': hazard_event, 'floodExtent': flood_extent, 'provenance': provenance}
FIXTURE_PATH = WORKDIR / 'hazard_event_fixture.json'
FIXTURE_PATH.write_text(json.dumps(fixture, indent=2))
print(FIXTURE_PATH)

## Outputs

- `flood_extent.geojson`: model-derived water/flood polygons.
- `hazard_event_fixture.json`: the `HazardEvent`, GeoJSON, and full provenance record.

Only accept and distribute these files after the mask overlay has been manually checked for incorrect band order, cloud/no-data artefacts, and AOI alignment.